# Day 024 — Exercise 4: scrape_and_extract

**What you'll build:** `scrape_and_extract(url, fields, model)` — the full pipeline in one function: fetch the page → clean the HTML → extract fields with the LLM.

**Why it matters:** This is the core of AI-powered scraping. Three helper functions (provided below) do the heavy lifting; your job is to wire them together in the right order. Any URL, any field list — one function call.

In [ ]:
import re
import json
import requests
import ollama
from bs4 import BeautifulSoup

## Provided Helpers

In [ ]:
def clean_html_text(html_string: str) -> str:
    soup = BeautifulSoup(html_string, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_schema_fields(
    text: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    fields_json = json.dumps(fields)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a structured data extractor. "
                    f"Extract the following fields from the text: {fields_json}. "
                    "Return JSON with exactly these keys. "
                    "Use null for any field you cannot find. "
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": f"Extract from this text:\n\n{text[:3000]}",
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        return json.loads(raw)
    except Exception:
        return {f: None for f in fields}

## Your Implementation

In [ ]:
def scrape_and_extract(
    url: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    """
    Fetch a URL, clean the HTML, and extract fields using the LLM.

    Args:
        url:    URL to scrape.
        fields: Field names to extract (e.g. ['title', 'description']).
        model:  Ollama model name.

    Returns:
        Dict with the extracted field values.

    Raises:
        requests.exceptions.HTTPError or ConnectionError on bad URLs.
    """
    # TODO: response = requests.get(url, timeout=10)
    # TODO: response.raise_for_status()
    # TODO: text = clean_html_text(response.text)
    # TODO: return extract_schema_fields(text, fields, model=model)
    pass

## Check Your Work

In [ ]:
SCRAPE_URL = "https://books.toscrape.com"
FIELDS = ['site_name', 'description']


def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'scrape_and_extract' in globals()
        passed += 1; print('\u2705 Check 1: scrape_and_extract defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    result = None

    # Check 2: returns a dict (1 network + 1 LLM call)
    try:
        result = scrape_and_extract(SCRAPE_URL, FIELDS)
        assert isinstance(result, dict), f'expected dict, got {type(result)}'
        passed += 1; print('\u2705 Check 2: returns a dict')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: dict has all requested field keys
    try:
        assert result is not None, 'result is None (Check 2 failed)'
        for f in FIELDS:
            assert f in result, f"missing field '{f}': {list(result)}"
        passed += 1; print(f'\u2705 Check 3: dict has all {len(FIELDS)} requested fields')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: at least one extracted value is a non-null string
    try:
        assert result is not None, 'result is None'
        non_null = [v for v in result.values() if v is not None]
        assert len(non_null) >= 1, \
            f'all fields are null — is the LLM running? result: {result}'
        passed += 1; print(f'\u2705 Check 4: {len(non_null)}/{len(FIELDS)} fields have values')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: raises on unreachable URL
    raised = False
    try:
        scrape_and_extract('http://localhost:9999/', FIELDS)
    except Exception:
        raised = True
    try:
        assert raised, 'scrape_and_extract should raise on unreachable URL'
        passed += 1; print('\u2705 Check 5: raises on unreachable URL')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def scrape_and_extract(
    url: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    text = clean_html_text(response.text)
    return extract_schema_fields(text, fields, model=model)
```

</details>